# Holdout-Evaluierung: letztes Rennen

Dieses Notebook testet das Modell auf dem zeitlich letzten Rennen im Datensatz, das im Training bewusst außen vor blieb.

**Ablauf**
- Laden von `driver_race_features.csv` und dem neu trainierten `final_model_logreg.joblib` (ohne das letzte Rennen).
- Holdout-Auswahl: identifiziert automatisch das letzte Rennen (maximales `race_date`) und nutzt nur diese Zeilen fürs Scoring.
- Encoding: Feature-Raum wie im Training nachbauen (One-Hot), Spalten exakt mit dem Modellschema abgleichen.
- Prognose: Top-10-Wahrscheinlichkeiten (`p_top10`) für alle Starts im Holdout-Rennen berechnen.
- Vergleich: `p_top10` neben dem echten `target_top10` ansehen, um die Generalisierung auf das unbekannte Rennen zu bewerten.

**Hinweise**
- `data/processed/holdout_latest_race.csv` muss gefüllt sein; das Modell aus Notebook 03 muss nach dem Holdout neu trainiert sein.
- Fokus nur auf das letzte Rennen; alle anderen Rennen verbleiben im Trainingsset.


In [62]:
import pandas as pd
import joblib
from pathlib import Path

FEATURES_PATH = Path("../data/processed/driver_race_features.csv")
HOLDOUT_PATH = Path("../data/processed/holdout_latest_race.csv")
MODEL_PATH = Path("../data/processed/final_model_logreg.joblib")

if not HOLDOUT_PATH.exists():
    raise FileNotFoundError(f"Holdout file missing: {HOLDOUT_PATH}. Re-run notebook 03 after creating the holdout.")

features = pd.read_csv(FEATURES_PATH)
holdout_df = pd.read_csv(HOLDOUT_PATH)
model = joblib.load(MODEL_PATH)

if holdout_df.empty:
    raise ValueError(
        "Holdout file has 0 rows. Ensure the latest race exists in driver_race_features.csv, "
        "rerun notebook 03 to regenerate the holdout and retrain the model."
    )

print("Features shape:", features.shape)
print("Holdout shape:", holdout_df.shape)
print("Model:", type(model))

Features shape: (25121, 10)
Holdout shape: (20, 18)
Model: <class 'sklearn.pipeline.Pipeline'>


In [63]:
# Define columns and holdout mask based on the holdout file
if holdout_df.shape[0] == 0:
    raise ValueError("Holdout is empty. Regenerate in notebook 03.")

uniq_races = holdout_df[['season','round','raceId']].drop_duplicates()
if len(uniq_races) != 1:
    raise ValueError(f"Expected one holdout race, found {len(uniq_races)}")

HOLDOUT_SEASON = int(uniq_races.iloc[0]['season'])
HOLDOUT_ROUND = int(uniq_races.iloc[0]['round'])
HOLDOUT_RACEID = uniq_races.iloc[0]['raceId']

mask = (
    (features['season'] == HOLDOUT_SEASON)
    & (features['round'] == HOLDOUT_ROUND)
    & (features['raceId'] == HOLDOUT_RACEID)
)

drop_cols = [
    "target_top10",
    "race_date",
    "grid",
    "driver_name",
    "constructor_name",
    "circuit_name",
    "raceId",
]
cat_cols = [c for c in ["grid_bucket"] if c in features.columns]


KeyError: 'circuitId'

In [ ]:
# Build training feature space (same as training notebook, without holdout)
train_X = features.loc[~mask].drop(columns=drop_cols, errors="ignore")
train_X = pd.get_dummies(train_X, columns=cat_cols, drop_first=True)
train_cols = train_X.columns
print("Train feature space:", train_X.shape)
print("Cat cols used:", cat_cols)

Train feature space: (25101, 91)
Cat cols used: ['circuitId', 'era', 'grid_bucket']


In [ ]:
# Prepare holdout features and align columns
X_new = holdout_df.drop(columns=drop_cols, errors="ignore")
X_new = pd.get_dummies(X_new, columns=cat_cols, drop_first=True)

for col in train_cols:
    if col not in X_new:
        X_new[col] = 0
X_new = X_new[train_cols]

print("Holdout matrix:", X_new.shape)
print("Cat cols used:", cat_cols)

Holdout matrix: (20, 91)
Cat cols used: ['circuitId', 'era', 'grid_bucket']


In [ ]:
# Predict probabilities and compare with actual target
proba = model.predict_proba(X_new)[:, 1]

# Use whatever identifier columns exist
display_cols = [c for c in [
    "driver_name", "constructor_name", "driverId", "constructorId",
    "grid", "grid_bucket", "target_top10"
] if c in holdout_df.columns]

preds = holdout_df[display_cols].copy()
preds["p_top10"] = proba
preds = preds.sort_values("p_top10", ascending=False).reset_index(drop=True)

preds

,driver_name,constructor_name,grid,target_top10,p_top10
0,Lando Norris,McLaren,1,1,0.841767
1,Carlos Sainz,Ferrari,3,1,0.834840
2,Max Verstappen,Red Bull,4,1,0.795123
3,Oscar Piastri,McLaren,2,1,0.792194
4,Fernando Alonso,Aston Martin,8,1,0.783334
5,Valtteri Bottas,Sauber,9,0,0.692227
6,Sergio Pérez,Red Bull,10,0,0.680803
7,George Russell,Mercedes,6,1,0.660935
8,Pierre Gasly,Alpine F1 Team,5,1,0.648063
9,Nico Hülkenberg,Haas F1 Team,7,1,0.602018
